### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="musk",
    dataset_year="1994",
    domain_str="chemistry & material science",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C51608",
    download_description="""
wget https://archive.ics.uci.edu/static/public/75/musk+version+2.zip && unzip musk+version+2.zip clean2.data.Z && uncompress clean2.data.Z && rm musk+version+2.zip && mkdir -p local-data-warehouse/musk && mv clean2.data local-data-warehouse/musk/
""",
    # References
    academic_reference_bibtex="""@article{dietterich1993comparison,
  title={A comparison of dynamic reposing and tangent distance for drug activity prediction},
  author={Dietterich, Thomas and Jain, Ajay and Lathrop, Richard and Lozano-Perez, Tomas},
  journal={Advances in neural information processing systems},
  volume={6},
  year={1993}
}
""",
    academic_reference_bibtex_key="dietterich1993comparison",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
- We rename the molecule IDs to remove the target leakage from the names.
- We drop the conformation name as it leaks information that the real task should not have (the correlation between specific conformations across samples).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="class",
    group_on="molecule_name",
    group_labels="per_group",
)

## Preprocessing

In [2]:
import pandas as pd
import uuid

df = pd.read_csv(dataset_mold.path / "clean2.data", header=None, names=[
    "molecule_name", "conformation_name", *[f"feature_{i}" for i in range(166)], "class",
])
print("Loaded data shape:", df.shape)

df = df.drop(columns=["conformation_name"])

df["class"] = df["class"].map({0: "non-musk", 1: "musk"})

# Create mapping: molecule -> random string id
mapping = {val: uuid.uuid4().hex[:12] for val in df["molecule_name"].unique()}
df["molecule_name"] = df["molecule_name"].map(mapping)

as_cat_type = ["molecule_name", "class"]
df[as_cat_type] = df[as_cat_type].astype("category")


df = df.sample(frac=1, random_state=42).sort_values(by="molecule_name").reset_index(drop=True)

Loaded data shape: (6598, 169)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 6,598
Columns: 168
Use sampling: False (sample size: 6,598)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['feature_126', 'feature_72', 'feature_33', 'feature_20', 'feature_132', 'feature_55', 'feature_128', 'feature_139', 'feature_59', 'feature_47']
Rows remaining as candidates after top-10 filter: 860 (of 6,598)

#### Duplicate Report
Total duplicate rows: 17 (0.26% of dataset)
Duplicate rows ignoring target: 17 (0.26% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,molecule_name,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_79,feature_80,feature_81,feature_82,feature_83,feature_84,feature_85,feature_86,feature_87,feature_88,feature_89,feature_90,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100,feature_101,feature_102,feature_103,feature_104,feature_105,feature_106,feature_107,feature_108,feature_109,feature_110,feature_111,feature_112,feature_113,feature_114,feature_115,feature_116,feature_117,feature_118,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128,feature_129,feature_130,feature_131,feature_132,feature_133,feature_134,feature_135,feature_136,feature_137,feature_138,feature_139,feature_140,feature_141,feature_142,feature_143,feature_144,feature_145,feature_146,feature_147,feature_148,feature_149,feature_150,feature_151,feature_152,feature_153,feature_154,feature_155,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,class
0,08d232c92c17,41,-198,-159,-47,-117,15,43,-79,-29,-74,-209,-124,-92,-201,-173,-295,32,-70,-91,-118,-89,-13,-58,12,105,9,-22,-82,23,115,-117,56,-29,70,-131,64,-112,2,-140,-40,-184,105,-74,-193,-196,-265,-119,-84,-88,-103,-99,-15,-16,-110,165,-63,51,-151,12,62,-101,-151,24,103,-139,57,-128,-108,-123,-116,-169,-128,-147,-174,-159,-170,4,-182,-101,-69,-83,7,-45,78,14,36,-141,-71,13,83,-202,69,-114,-76,28,-39,-52,41,-155,28,-206,133,-165,-172,-193,-220,-173,-96,-160,-193,-155,-56,-65,-51,22,66,129,60,66,-27,94,147,107,-108,-107,-10,14,94,-33,-187,17,-138,-89,-133,-148,-102,45,-102,-46,-55,-33,-37,17,20,-178,-102,-119,-124,-135,-27,-67,-123,-112,-121,98,-77,104,53,123,55,127,143,152,-60,-127,57,musk
1,08d232c92c17,42,-190,-138,-81,-117,-37,67,-99,97,-26,-190,-162,-139,-199,-192,-303,59,-123,-131,-73,77,111,136,148,93,-26,60,-94,-98,99,-116,58,-25,39,-132,65,-168,15,-128,54,-150,104,-47,-176,-213,-307,-88,27,-88,-18,-76,127,127,-130,103,-77,98,-135,-80,14,-102,-153,24,76,-150,57,-166,-61,-114,-30,-147,-130,-181,-187,-80,-182,5,-177,-159,-65,138,75,23,71,-32,40,-65,-177,-78,89,-202,72,-116,-153,29,-41,-125,38,-156,86,-164,135,-156,-176,-196,-245,-171,-128,-148,-205,-138,133,174,167,-45,-5,164,26,84,34,74,47,50,-106,-104,-8,-3,90,-50,-187,17,-138,-55,-136,-166,-75,63,-52,92,-32,4,20,55,25,-178,-103,-120,-39,-56,67,-65,-125,-114,-111,99,-89,-237,-74,22,52,127,144,147,-60,-123,55,musk
2,08d232c92c17,41,-198,-149,-31,-117,50,66,-72,-26,7,-197,-104,-148,-193,-197,-304,34,-152,-107,-63,-49,-11,-78,307,101,-27,23,-126,-48,102,-117,55,-30,18,-131,64,-158,-35,-139,28,-180,104,-47,-176,-196,-308,-102,-63,-80,-5,-67,-30,-23,-120,111,-56,86,-153,-9,17,-100,-153,24,113,-132,57,-166,-104,-123,-72,-162,-129,-194,-164,-133,-180,4,-169,-114,-53,-58,33,-53,147,36,34,-123,-173,-13,131,-202,68,-116,-228,28,-36,-40,1,-155,79,-203,133,-175,-182,-195,-197,-119,-47,-166,-192,-142,-50,-67,-54,46,10,186,25,76,-10,82,67,106,-109,-108,-11,-26,84,-137,-187,18,-138,-81,-110,-171,-133,-5,-122,-75,-52,-71,-1,57,10,-178,-41,-116,-85,-106,35,-67,-125,-113,-123,98,-84,-242,-295,67,55,128,143,151,-60,-1

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,molecule_name,category,0.0,0.0,102.0,"d413b1172202, 46baa131aa04, fbbe4fc3b136, 54d81e62e65b, d3fe9a4fc778, bfa628a045f0, 0dd837278802, 38f40cf0fb98, e5cc89ea6bad, 170f26baa93c"
1,class,category,0.0,0.0,2.0,"non-musk, musk"
2,feature_0,int64,0.0,0.0,202.0,"44, 43, 35, 36, 46, 51, 37, 48, 47, 57"
3,feature_1,int64,0.0,0.0,260.0,"-198, -194, -199, -193, -195, -192, -196, -197, 86, -191"
4,feature_2,int64,0.0,0.0,221.0,"-145, -144, -112, -19, -22, -111, 31, -62, -23, -146"
5,feature_3,int64,0.0,0.0,257.0,"-76, -77, -69, 28, -70, 29, 33, 131, 32, 152"
6,feature_4,int64,0.0,0.0,129.0,"-117, -116, -115, -113, -112, -111, -114, -108, -110, -109"
7,feature_5,int64,0.0,0.0,358.0,"11, 10, 12, 86, 85, 54, 55, -154, 53, 52"
8,feature_6,int64,0.0,0.0,323.0,"56, 26, 57, -163, 27, -160, -162, -161, -164, -159"
9,feature_7,int64,0.0,0.0,389.0,"-95, -96, -103, 57, 64, -3, -171, -102, 67, -5"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
feature_0,6598.0,58.945135,53.249007,-31.0,292.0
feature_1,6598.0,-119.128524,90.813375,-199.0,95.0
feature_2,6598.0,-73.146560,67.956235,-167.0,81.0
feature_3,6598.0,-0.628372,80.444617,-114.0,161.0
feature_4,6598.0,-103.533495,64.387559,-118.0,325.0
feature_5,6598.0,18.359806,80.593655,-183.0,200.0
feature_6,6598.0,-14.108821,115.315673,-171.0,220.0
feature_7,6598.0,-1.858290,90.372537,-225.0,320.0
feature_8,6598.0,-86.003031,108.326676,-245.0,147.0
feature_9,6598.0,-44.495756,72.088903,-286.0,231.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column        rank                            
class         1         non-musk   5581  84.59
              2             musk   1017  15.41
molecule_name 1     d413b1172202   1044  15.82
              2     46baa131aa04   1010  15.31
              3     fbbe4fc3b136    911  13.81
              4     54d81e62e65b    383   5.80
              5     d3fe9a4fc778    344   5.21

In [8]:
# Target Distribution
target_df

,count,pct
class,,
non-musk,5581,84.59
musk,1017,15.41


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Providing recommendations based on number of groups (102).
Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create stratified grouped 20-repeated 3-fold split. This creates ca. 30 group members (500-3000 samples) per test set.",
    splits=splits
)

Using Stratified Grouped splits.
Using label-per-group grouped splits.
Creating index-based splits for 102 groups
Using Stratified IID splits.


Repeat 0, Fold 0:
            Train N: 4672, Test N: 1926
            Target Distribution:
            	Train target distribution: {'non-musk': 0.8180650684931506, 'musk': 0.1819349315068493}
            	Test target distribution: {'non-musk': 0.9132917964693665, 'musk': 0.08670820353063344}
            Group Distribution molecule_name:
            	Train: 68
            	Test: 34
            
Repeat 0, Fold 1:
            Train N: 5831, Test N: 767
            Target Distribution:
            	Train target distribution: {'non-musk': 0.8940147487566455, 'musk': 0.10598525124335448}
            	Test target distribution: {'musk': 0.5202086049543677, 'non-musk': 0.47979139504563234}
            Group Distribution molecule_name:
            	Train: 68
            	Test: 34
            
Repeat 0, Fold 2:
            Train N: 2693, Test N: 3905
            Target Distribution:
            	Train target distribution: {'non-musk': 0.7898254734496843, 'musk': 0.21017452655031563}
            	

Repeat 9, Fold 2:
            Train N: 4431, Test N: 2167
            Target Distribution:
            	Train target distribution: {'non-musk': 0.8724892800722185, 'musk': 0.12751071992778154}
            	Test target distribution: {'non-musk': 0.7914167051222889, 'musk': 0.20858329487771113}
            Group Distribution molecule_name:
            	Train: 68
            	Test: 34
            
Repeat 10, Fold 0:
            Train N: 4813, Test N: 1785
            Target Distribution:
            	Train target distribution: {'non-musk': 0.8607936837731145, 'musk': 0.1392063162268855}
            	Test target distribution: {'non-musk': 0.8056022408963586, 'musk': 0.19439775910364146}
            Group Distribution molecule_name:
            	Train: 68
            	Test: 34
            
Repeat 10, Fold 1:
            Train N: 4510, Test N: 2088
            Target Distribution:
            	Train target distribution: {'non-musk': 0.8596452328159645, 'musk': 0.14035476718403547}
          

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to musk/019d738d-ae72-7a0a-a2e4-6b797ab153fd


019d738d-ae72-7a0a-a2e4-6b797ab153fd
fdd24672548805b802a47ffd60b34961bd2c55c4d7aabd17d5e01c12a7d82c9d
